# Classification Summary Test

이 노트북은 `classification_detail` 또는 `classification_full` 결과를 `classification_summary` 테이블로 집계/저장하는 테스트용입니다.

실행 순서:
1. repo 최신 pull
2. summary writer import / reload
3. 대상 classification 입력 데이터 로드
4. summary 저장
5. 저장 결과 재조회 및 분포 확인


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)


In [ ]:
%sh
cd /Workspace/Users/jungryo.lee@lge.com/prj_TV_voc && git pull

In [ ]:
import common.config_loader as config_loader
import taxonomy.classification_summary_writer as classification_summary_writer

importlib.reload(config_loader)
importlib.reload(classification_summary_writer)

from common.config_loader import get_output_table, load_config
from taxonomy.classification_summary_writer import save_classification_summary

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")

# classification_detail: memo_id 기준 샘플/분류 검증용
# classification_full: 원본 row 기준 운영 분포용
CLASSIFICATION_INPUT_TABLE_KEY = "classification_full"
CLASSIFICATION_INPUT_TABLE = get_output_table(config, CLASSIFICATION_INPUT_TABLE_KEY)
CLASSIFICATION_SUMMARY_TABLE = get_output_table(config, "classification_summary")


In [ ]:
MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-06. 리모컨 사용성"
TARGET_SC_MEASUREMENT = 1

detail_df = (
    spark.table(CLASSIFICATION_INPUT_TABLE)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
)

detail_cnt = detail_df.count()
print({"input_table_key": CLASSIFICATION_INPUT_TABLE_KEY, "detail_cnt": detail_cnt})

if detail_cnt == 0:
    raise ValueError(f"No rows found in {CLASSIFICATION_INPUT_TABLE_KEY} for the target group.")

In [ ]:
saved_table = save_classification_summary(
    spark=spark,
    config=config,
    detail_df=detail_df,
    write_mode="replace_groups",
)

print("saved_table =", saved_table)

In [ ]:
summary_df = (
    spark.table(CLASSIFICATION_SUMMARY_TABLE)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
)

display(
    summary_df.orderBy(F.col("memo_cnt").desc(), F.col("pred_topic").asc())
)

In [ ]:
display(
    summary_df.select(
        "pred_topic",
        "pred_topic_type",
        "memo_cnt",
        "topic_ratio_pct",
        "group_total_cnt",
        "overall_cnt",
        "overall_ratio",
        "others_cnt",
        "others_ratio",
        "llm_used_cnt",
        "llm_used_ratio",
        "review_needed_cnt",
        "review_needed_ratio",
    )
    .orderBy(F.col("memo_cnt").desc(), F.col("pred_topic").asc())
)

In [ ]:
display(
    detail_df.select(
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(30)
)